# DML Lab Assignment No. 6
## Prediction Model using Pre-Trained Deep Learning Models (Transfer Learning)
**Pre-trained Model Used:** ResNet50 (fine-tuned on CIFAR-10)

### Step 1: Import Libraries and Load Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print('TensorFlow version:', tf.__version__)

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

### Step 2: Normalize and Resize Dataset
ResNet50 expects at least 32×32; we resize to 64×64 for better feature extraction.

In [ ]:
IMG_SIZE = 64

def preprocess(X, y):
    X = tf.image.resize(X, (IMG_SIZE, IMG_SIZE)).numpy()
    X = tf.keras.applications.resnet50.preprocess_input(X)
    y = to_categorical(y, 10)
    return X, y

X_train_p, y_train_p = preprocess(X_train, y_train)
X_test_p,  y_test_p  = preprocess(X_test,  y_test)

print('Processed X_train:', X_train_p.shape)
print('Processed X_test:', X_test_p.shape)

### Step 3: Load Pre-trained ResNet50 (without top layer)

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,           # Remove original classification head
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze all base model layers initially
base_model.trainable = False
print(f'Base model layers: {len(base_model.layers)}')
print('Base model frozen for feature extraction.')

### Step 4: Fine-Tune — Add Custom Classification Head

In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')   # CIFAR-10 classes
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

### Step 5A: Train — Phase 1 (Feature Extraction, frozen base)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

history1 = model.fit(
    X_train_p, y_train_p,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)
print('Phase 1 complete.')

### Step 5B: Train — Phase 2 (Fine-tuning, unfreeze last 30 layers)

In [ ]:
# Unfreeze last 30 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),   # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    X_train_p, y_train_p,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)
print('Phase 2 fine-tuning complete.')

### Step 6: Predict / Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(X_test_p, y_test_p, verbose=0)
print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss:     {test_loss:.4f}')

# Show sample predictions
preds = model.predict(X_test_p[:10])
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    # Reverse ResNet preprocessing for display
    img = X_test[i]  # original image
    plt.imshow(img)
    pred = class_names[np.argmax(preds[i])]
    true = class_names[y_test[i][0]]
    color = 'green' if pred == true else 'red'
    plt.title(f'P:{pred}\nT:{true}', color=color, fontsize=8)
    plt.axis('off')
plt.suptitle('ResNet50 Predictions')
plt.tight_layout()
plt.show()

### Step 7: Performance Visualization

In [ ]:
# Combine both training phases
acc  = history1.history['accuracy']  + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss']      + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(acc,     label='Train Accuracy')
axes[0].plot(val_acc, label='Val Accuracy')
axes[0].axvline(x=len(history1.history['accuracy'])-1,
                color='gray', linestyle='--', label='Fine-tune start')
axes[0].set_title('ResNet50 Accuracy (Transfer Learning)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(loss,     label='Train Loss')
axes[1].plot(val_loss, label='Val Loss')
axes[1].axvline(x=len(history1.history['loss'])-1,
                color='gray', linestyle='--', label='Fine-tune start')
axes[1].set_title('ResNet50 Loss (Transfer Learning)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()